# Demo 1: LTC 液态神经元核心原理

本 Notebook 从零实现一个液态时间常数（LTC）神经元，直观展示其核心机制：
1. ODE 驱动的连续时间状态演化
2. 动态时间常数如何随输入变化
3. 与传统固定时间常数神经元的对比

---

## 1. 核心公式回顾

LTC 神经元的状态演化由以下 ODE 描述：

$$\frac{dx(t)}{dt} = -\frac{1}{\tau_{\text{eff}}} \odot x(t) + \frac{1}{\tau_{\text{eff}}} \odot A$$

其中有效时间常数：

$$\tau_{\text{eff}} = \tau + NN(x(t), I(t), \theta)$$

- $x(t)$：神经元隐藏状态
- $I(t)$：外部输入
- $\tau$：基础时间常数
- $NN(\cdot)$：输入依赖的非线性函数（使时间常数动态化）
- $A$：目标激活值

**关键洞察**：当 $\tau_{\text{eff}}$ 大时，状态变化慢（"记忆长"）；当 $\tau_{\text{eff}}$ 小时，状态变化快（"反应快"）。这种动态调节是 LNN 的核心。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 2. 最小化 LTC 神经元实现

我们用一个简化的单神经元模型来理解核心动力学。

In [ ]:
class LiquidNeuron:
    """
    简化版 LTC 神经元
    dx/dt = -(1/tau_eff) * x + (1/tau_eff) * A
    tau_eff = tau + sigmoid(w * I + b)  # 输入依赖的动态时间常数
    """
    def __init__(self, tau=1.0, w=2.0, b=0.0, A=1.0):
        self.tau = tau
        self.w = w
        self.b = b
        self.A = A

    def tau_eff(self, I):
        return self.tau + 1.0 / (1.0 + np.exp(-(self.w * I + self.b)))

    def derivative(self, x, t, I_func):
        I = I_func(t)
        te = self.tau_eff(I)
        return -(1.0 / te) * x + (1.0 / te) * self.A

    def simulate(self, I_func, t_span, x0=0.0):
        sol = odeint(self.derivative, x0, t_span, args=(I_func,))
        return sol.flatten()


class FixedNeuron:
    """
    传统固定时间常数神经元（对照）
    dx/dt = -(1/tau) * x + (1/tau) * A
    """
    def __init__(self, tau=1.0, A=1.0):
        self.tau = tau
        self.A = A

    def derivative(self, x, t, I_func):
        return -(1.0 / self.tau) * x + (1.0 / self.tau) * self.A

    def simulate(self, I_func, t_span, x0=0.0):
        sol = odeint(self.derivative, x0, t_span, args=(I_func,))
        return sol.flatten()

## 3. 实验：动态时间常数 vs 固定时间常数

输入信号包含一个阶跃变化，观察两种神经元的响应差异。

In [ ]:
def step_input(t, t_step=5.0, low=0.2, high=1.0):
    return high if t >= t_step else low

t_span = np.linspace(0, 15, 1000)

liquid = LiquidNeuron(tau=1.0, w=3.0, b=-1.0, A=1.0)
fixed = FixedNeuron(tau=1.5, A=1.0)

x_liquid = liquid.simulate(step_input, t_span, x0=0.0)
x_fixed = fixed.simulate(step_input, t_span, x0=0.0)

I_values = np.array([step_input(t) for t in t_span])
tau_values = np.array([liquid.tau_eff(I) for I in I_values])

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(t_span, I_values, 'k-', linewidth=2, label='Input I(t)')
axes[0].axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Input')
axes[0].set_title('Input Signal (Step at t=5)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_span, x_liquid, 'b-', linewidth=2, label='Liquid Neuron (dynamic τ)')
axes[1].plot(t_span, x_fixed, 'r--', linewidth=2, label='Fixed Neuron (τ=1.5)')
axes[1].axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('State x(t)')
axes[1].set_title('Neuron State Response')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_span, tau_values, 'g-', linewidth=2, label='τ_eff (Liquid)')
axes[2].axhline(y=1.5, color='r', linestyle='--', linewidth=2, label='τ (Fixed)')
axes[2].axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Time Constant')
axes[2].set_title('Effective Time Constant (Dynamic vs Fixed)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('demo1_step_response.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observation: Liquid neuron adapts its response speed based on input.")
print(f"  Before step: τ_eff = {liquid.tau_eff(0.2):.3f} (slower decay, longer memory)")
print(f"  After step:  τ_eff = {liquid.tau_eff(1.0):.3f} (faster response)")

## 4. 实验：正弦输入下的动态适应

LTC 神经元在不同频率的输入下会自动调整时间常数，实现自适应滤波。

In [ ]:
def sine_input(t, freq=0.5, amp=1.0):
    return amp * (0.5 + 0.5 * np.sin(2 * np.pi * freq * t))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, freq in enumerate([0.1, 0.5, 1.0, 2.0]):
    row, col = idx // 2, idx % 2
    ax = axes[row][col]

    I_func = lambda t, f=freq: sine_input(t, freq=f)
    x_liquid = liquid.simulate(I_func, t_span, x0=0.0)
    x_fixed = fixed.simulate(I_func, t_span, x0=0.0)
    I_vals = np.array([I_func(t) for t in t_span])

    ax.plot(t_span, I_vals, 'k-', alpha=0.3, linewidth=1, label='Input')
    ax.plot(t_span, x_liquid, 'b-', linewidth=2, label='Liquid')
    ax.plot(t_span, x_fixed, 'r--', linewidth=2, label='Fixed')
    ax.set_title(f'Sine Input freq={freq} Hz')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('LTC Neuron vs Fixed Neuron under Different Input Frequencies', fontsize=14)
plt.tight_layout()
plt.savefig('demo1_sine_response.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 可视化：时间常数的动态变化轨迹

在相空间中观察 $\tau_{\text{eff}}$ 随输入和状态的变化。

In [ ]:
I_range = np.linspace(0, 2, 200)
tau_eff_range = np.array([liquid.tau_eff(i) for i in I_range])

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(I_range, tau_eff_range, 'b-', linewidth=2.5)
ax.axhline(y=1.5, color='r', linestyle='--', linewidth=2, label='Fixed τ=1.5')
ax.fill_between(I_range, tau_eff_range, 1.5, alpha=0.15, color='blue')
ax.set_xlabel('Input I', fontsize=12)
ax.set_ylabel('Effective Time Constant τ_eff', fontsize=12)
ax.set_title('Dynamic τ_eff vs Input (Liquid Neuron)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.annotate('τ_eff > τ: slower response\n(longer memory)',
            xy=(0.3, 1.65), fontsize=10, color='blue')
ax.annotate('τ_eff ≈ τ: faster response\n(quick adaptation)',
            xy=(1.2, 1.55), fontsize=10, color='blue')
plt.tight_layout()
plt.savefig('demo1_tau_eff.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 关键结论

| 特性 | 固定时间常数 | 液态时间常数 (LTC) |
|------|-------------|-------------------|
| 响应速度 | 恒定 | 随输入动态调整 |
| 低输入时 | 固定速率衰减 | 更慢衰减（保持记忆） |
| 高输入时 | 固定速率响应 | 更快响应（快速适应） |
| 适应性 | 无 | 自适应滤波 |

**核心洞察**：LTC 神经元通过让时间常数成为输入的函数，实现了"该快时快，该慢时慢"的自适应行为——这正是线虫仅用 302 个神经元就能灵活应对环境变化的秘诀。

---

Next: [Demo 2: CfC 闭式解与 LTC 对比](./demo2_cfc_vs_ltc.ipynb)